In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data/raw")

train_df = pd.read_csv(
    DATA_DIR / "UNSW_NB15_training-set.csv",
    low_memory=False
)

train_df.columns = train_df.columns.str.strip()

X = train_df.drop(
    columns=["id", "label", "attack_cat"]
)
y = train_df["label"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training subset:", X_train.shape)
print("Validation set:", X_val.shape)

print("\nTraining distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True).round(3))


Training subset: (140272, 42)
Validation set: (35069, 42)

Training distribution:
label
1    0.681
0    0.319
Name: proportion, dtype: float64

Validation distribution:
label
1    0.681
0    0.319
Name: proportion, dtype: float64


In [2]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

categorical_columns = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=np.number
).columns.tolist()


def build_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])


def build_model(classifier):
    return Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", classifier)
    ])


def evaluate_model(name, model, X_eval, y_eval):
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_eval, predictions, labels=[0, 1]
    ).ravel()

    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_eval, predictions),
        "Attack Precision": precision_score(y_eval, predictions, zero_division=0),
        "Attack Recall": recall_score(y_eval, predictions, zero_division=0),
        "Attack F1": f1_score(y_eval, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities),
        "False Positive Rate": fp / (fp + tn) if (fp + tn) else 0,
        "False Negatives": fn,
        "False Positives": fp,
    }

    print(f"\n{name}")
    print(classification_report(
        y_eval,
        predictions,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        digits=4,
        zero_division=0
    ))

    return results


In [3]:

results = []
trained_models = {}

log_model = build_model(
    LogisticRegression(max_iter=1000)
)

start = time.perf_counter()
log_model.fit(X_train, y_train)
log_time = time.perf_counter() - start

trained_models["Logistic Regression"] = log_model

log_result = evaluate_model(
    "Logistic Regression",
    log_model,
    X_val,
    y_val
)
log_result["Training Time (seconds)"] = log_time
results.append(log_result)

print(f"Training time: {log_time:.2f} seconds")



Logistic Regression
              precision    recall  f1-score   support

      Normal     0.9706    0.8260    0.8925     11200
      Attack     0.9237    0.9883    0.9549     23869

    accuracy                         0.9364     35069
   macro avg     0.9472    0.9071    0.9237     35069
weighted avg     0.9387    0.9364    0.9350     35069

Training time: 10.56 seconds


In [4]:

rf_model = build_model(
    RandomForestClassifier(
        n_estimators=150,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)

start = time.perf_counter()
rf_model.fit(X_train, y_train)
rf_time = time.perf_counter() - start

trained_models["Random Forest"] = rf_model

rf_result = evaluate_model(
    "Random Forest",
    rf_model,
    X_val,
    y_val
)
rf_result["Training Time (seconds)"] = rf_time
results.append(rf_result)

print(f"Training time: {rf_time:.2f} seconds")



Random Forest
              precision    recall  f1-score   support

      Normal     0.9157    0.9473    0.9312     11200
      Attack     0.9749    0.9591    0.9669     23869

    accuracy                         0.9553     35069
   macro avg     0.9453    0.9532    0.9491     35069
weighted avg     0.9560    0.9553    0.9555     35069

Training time: 48.35 seconds


In [5]:

from xgboost import XGBClassifier

xgb_model = build_model(
    XGBClassifier(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )
)

start = time.perf_counter()
xgb_model.fit(X_train, y_train)
xgb_time = time.perf_counter() - start

trained_models["XGBoost"] = xgb_model

xgb_result = evaluate_model(
    "XGBoost",
    xgb_model,
    X_val,
    y_val
)
xgb_result["Training Time (seconds)"] = xgb_time
results.append(xgb_result)

print(f"Training time: {xgb_time:.2f} seconds")



XGBoost
              precision    recall  f1-score   support

      Normal     0.9474    0.9220    0.9345     11200
      Attack     0.9638    0.9760    0.9699     23869

    accuracy                         0.9587     35069
   macro avg     0.9556    0.9490    0.9522     35069
weighted avg     0.9586    0.9587    0.9586     35069

Training time: 7.54 seconds


In [6]:

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        by="Attack F1",
        ascending=False
    ).round(4)
)


,Model,Accuracy,Balanced Accuracy,Attack Precision,Attack Recall,Attack F1,ROC-AUC,False Positive Rate,False Negatives,False Positives,Training Time (seconds)
2,XGBoost,0.9587,0.9490,0.9638,0.9760,0.9699,0.9937,0.0780,573,874,7.5409
1,Random Forest,0.9553,0.9532,0.9749,0.9591,0.9669,0.9938,0.0527,977,590,48.3472
0,Logistic Regression,0.9364,0.9071,0.9237,0.9883,0.9549,0.9843,0.1740,280,1949,10.5607


In [7]:

best_name = max(
    trained_models,
    key=lambda name: roc_auc_score(
        y_val,
        trained_models[name].predict_proba(X_val)[:, 1]
    )
)

best_model = trained_models[best_name]
val_probabilities = best_model.predict_proba(X_val)[:, 1]

print("Highest validation ROC-AUC:", best_name)


Highest validation ROC-AUC: Random Forest


In [8]:

threshold_results = []

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    predictions = (val_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Attack Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Attack Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "Attack F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "False Positive Rate": fp / (fp + tn),
        "False Negatives": fn,
        "False Positives": fp
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))


,Threshold,Attack Precision,Attack Recall,Attack F1,False Positive Rate,False Negatives,False Positives
0,0.2,0.9299,0.9972,0.9624,0.1603,67,1795
1,0.3,0.9476,0.9894,0.9680,0.1167,252,1307
2,0.4,0.9646,0.9770,0.9707,0.0765,549,857
3,0.5,0.9749,0.9591,0.9669,0.0527,977,590
4,0.6,0.9837,0.9384,0.9605,0.0330,1471,370
5,0.7,0.9902,0.9128,0.9499,0.0193,2081,216
6,0.8,0.9957,0.8781,0.9332,0.0081,2909,91


In [9]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

xgb_model = trained_models["XGBoost"]

xgb_probabilities = xgb_model.predict_proba(X_val)[:, 1]

threshold_results = []

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred = (xgb_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, y_pred, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Attack Precision": precision_score(y_val, y_pred, zero_division=0),
        "Attack Recall": recall_score(y_val, y_pred, zero_division=0),
        "Attack F1": f1_score(y_val, y_pred, zero_division=0),
        "False Positive Rate": fp / (fp + tn),
        "False Negatives": fn,
        "False Positives": fp
    })

xgb_threshold_df = pd.DataFrame(threshold_results)

display(xgb_threshold_df.round(4))

,Threshold,Attack Precision,Attack Recall,Attack F1,False Positive Rate,False Negatives,False Positives
0,0.2,0.9239,0.9987,0.9599,0.1753,31,1963
1,0.3,0.9368,0.9939,0.9645,0.1429,145,1601
2,0.4,0.9515,0.9872,0.9690,0.1071,306,1200
3,0.5,0.9638,0.9760,0.9699,0.0780,573,874
4,0.6,0.9736,0.9624,0.9679,0.0557,898,624
5,0.7,0.9819,0.9437,0.9624,0.0371,1345,416
6,0.8,0.9893,0.9172,0.9519,0.0211,1977,236


In [10]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

rf = trained_models["Random Forest"]
xgb = trained_models["XGBoost"]

rf_probs = rf.predict_proba(X_val)[:, 1]
xgb_probs = xgb.predict_proba(X_val)[:, 1]

# Start with equal weights
ensemble_probs = 0.5 * rf_probs + 0.5 * xgb_probs

ensemble_results = []

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    predictions = (ensemble_probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    ensemble_results.append({
        "Threshold": threshold,
        "Attack Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Attack Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "Attack F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "False Positive Rate": fp / (fp + tn),
        "False Negatives": fn,
        "False Positives": fp
    })

display(pd.DataFrame(ensemble_results).round(4))

,Threshold,Attack Precision,Attack Recall,Attack F1,False Positive Rate,False Negatives,False Positives
0,0.2,0.9246,0.9986,0.9601,0.1737,34,1945
1,0.3,0.9411,0.9929,0.9663,0.1325,169,1484
2,0.4,0.9585,0.9838,0.9710,0.0907,387,1016
3,0.5,0.9714,0.9704,0.9709,0.0610,706,683
4,0.6,0.9793,0.9516,0.9653,0.0429,1155,480
5,0.7,0.9868,0.9288,0.9569,0.0264,1699,296
6,0.8,0.9934,0.8987,0.9437,0.0127,2419,142


In [ ]:
thresholds = {
    "Random Forest": 0.4,
    "XGBoost": 0.5,
    "Weighted Ensemble": 0.5
}